## Huấn luyện mô hình để nhận dạng layout
Trong notebook sử dụng thư viện detectron2 để huấn luyện mô hình Faster-RCNN để nhận diện layout như id, name, address 

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import numpy as np
import json
import torch, torchvision
from PIL import ImageFile, Image, ImageOps
ImageFile.LOAD_TRUNCATED_IMAGES = True

import detectron2
from detectron2.utils.logger import setup_logger

# import some common libraries

import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import onnx
import shutil

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2.export import export_onnx_model, export_caffe2_model, add_export_config
from detectron2.structures import BoxMode

plt.rcParams["figure.figsize"] = (20,20)

%matplotlib inline

### Dataset
Khởi tạo dataset theo yêu cầu của thư viện detectron2 

In [ ]:
def get_data_dicts(img_dir, json_file):
    with open(json_file) as f:
        imgs_anns = json.load(f)
        classes = meta['region']['type']['options']
        class2idx = {c:i for i, c in enumerate(classes.values())}
        
        imgs_anns = imgs_anns['_via_img_metadata']
        imgs_anns = list(imgs_anns.values())
    print(classes, class2idx)
    dataset_dicts = []    
    for idx, e in enumerate(imgs_anns):
        
        try:
            record = {}
            objs = []
            
            filename = e['filename']                    
            filename = os.path.join(img_dir, filename) 
            height, width = cv2.imread(filename, 0).shape[:2]
            
            record["file_name"] = filename
            record["image_id"] = idx
            record["height"] = height
            record["width"] = width

            regions = e["regions"]
            
            for region in regions:
                
                try:
                    shape_attributes = region['shape_attributes']
                    region_attributes = region['region_attributes']
                    shape_type = shape_attributes['name']
                    x, y = shape_attributes['x'], shape_attributes['y']
                    w, h = shape_attributes['width'], shape_attributes['height']                
                    loai_thong_tin = classes[region_attributes['type']]
                    loai_thong_tin = class2idx[loai_thong_tin]
                    
                    x1, y1, x2, y2 = x, y, x+w, y+h
                    bbox = [x1, y1, x2, y2]            
                    poly = [x1, y1, x1, y2, x2, y2, x2, y1]

                    obj = {
                        "bbox": bbox,                            
                        "bbox_mode": BoxMode.XYXY_ABS,
                        "category_id": loai_thong_tin,
                        "iscrowd": 0
                    }
                    objs.append(obj)                
                except Exception as e:
                    print(e)
                                
        except Exception as e:            
            print(e)                    
            
        if len(objs) > 0:
            record["annotations"] = objs
            dataset_dicts.append(record)
            
    return dataset_dicts

In [ ]:
train_json_file = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/layout/train_annotation.json'
test_json_file = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/layout/test_annotation.json'
img_dir = '/data2/sftp/vmg/upload/nic/du_lieu_ban_giao/nic/layout/img/'

meta = json.load(open(train_json_file))
meta = meta['_via_attributes']

classes = list(meta['region']['type']['options'].values())

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog

for d, fname in zip(["val", "train"], [test_json_file, train_json_file]):
    DatasetCatalog.register(d, lambda fname=fname: get_data_dicts(img_dir, fname))
    MetadataCatalog.get(d).set(thing_classes=classes)
        
metadata = MetadataCatalog.get("train")

### Custom Trainer

In [ ]:
from detectron2.data import DatasetMapper, build_detection_train_loader, build_detection_test_loader
from detectron2.data import transforms as T
from detectron2.data import detection_utils as utils
import copy 

class CustomerDatasetMapper():
    def __init__(self, cfg, is_train):
        self.is_train = is_train
    
    def __call__(self, dataset_dict):
        
        # Implement a mapper, similar to the default DatasetMapper, but with your own customizations
        dataset_dict = copy.deepcopy(dataset_dict)  # it will be modified by code below
        image = utils.read_image(dataset_dict["file_name"], format="RGB")
        if self.is_train:
            image, transforms = T.apply_transform_gens([T.ResizeShortestEdge([1024, 1024]), T.RandomBrightness(0.9,1.1), T.RandomContrast(0.9, 1.1)], image)
        else:
            image, transforms = T.apply_transform_gens([T.ResizeShortestEdge([1024, 1024])], image)

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).astype("float32"))

        annos = [
            utils.transform_instance_annotations(obj, transforms, image.shape[:2])
            for obj in dataset_dict.pop("annotations")
            if obj.get("iscrowd", 0) == 0
        ]
        instances = utils.annotations_to_instances(annos, image.shape[:2])
        dataset_dict["instances"] = utils.filter_empty_instances(instances)
        return dataset_dict

class MyTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)
    
    @classmethod
    def build_train_loader(cls,cfg):
        return build_detection_train_loader(cfg, mapper=CustomerDatasetMapper(cfg, True))
    
    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        return build_detection_test_loader(cfg, dataset_name, mapper=CustomerDatasetMapper(cfg, False))

In [ ]:
dataset_dicts = get_data_dicts(img_dir, test_json_file)

### Visualize test set

In [ ]:
i = 0
for d in random.sample(dataset_dicts, 10):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5)
    visualizer._default_font_size = 30
    vis = visualizer.draw_dataset_dict(d)    
    fig = plt.figure(figsize = (20,20))
    plt.imshow(vis.get_image()[:, :, ::-1])
    plt.show()

### Set các tham số cho trainer

In [ ]:
cfg = get_cfg()
cfg.CLASSES = classes

cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("train",)
cfg.DATASETS.TEST = ("val",)
cfg.DATALOADER.NUM_WORKERS = 5
cfg.MODEL.WEIGHTS = '/var/www/html/nic/layout/model.pth'
cfg.SOLVER.IMS_PER_BATCH = 2

cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.434, 0.318, 0.27], [0.41, 0.223, 0.199], [0.329, 0.177, 0.288], [0.158, 0.269, 0.147], [0.153, 0.164, 0.156]]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[127, 153, 173], [246, 185, 199], [270, 212, 292], [223, 314, 238], [263, 302, 344]]



cfg.MODEL.BACKBONE.FREEZE_AT = 0

cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
# # cfg.SOLVER.MAX_ITER = 300000    # 
# cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256 
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(classes)  # only has one class (ballon)
cfg.TEST.EVAL_PERIOD = 7500
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 3000
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set the testing threshold for this model

### Bắt đầu huấn luyện

In [ ]:
shutil.rmtree(cfg.OUTPUT_DIR, ignore_errors=True)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = MyTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

### Lưu config để sử dụng lúc predict

In [ ]:
with open('config.yml', 'w') as f:
    f.write(cfg.dump())